In [4]:
import pandas as pd

df1 = pd.read_json("normalized_products/dataset_1_normalized.json")
df2 = pd.read_json("normalized_products/dataset_2_normalized.json")
df3 = pd.read_json("normalized_products/dataset_3_normalized.json")
df4 = pd.read_json("normalized_products/dataset_4_normalized.json")
kb = pd.concat([df2, df3, df4], ignore_index=True)

# Common columns across all datasets
common_cols = set(df1.columns) & set(kb.columns)

print(f"Total common columns: {len(common_cols)}")
print()

# For each common column show: missing in D1, available in KB, sample values
print(f"{'Attribute':<35} {'Missing D1':>10} {'Avail KB':>10}  Sample values")
print("-" * 90)
for col in sorted(common_cols):
    missing = df1[col].isna().sum()
    available_kb = kb[col].notna().sum()
    sample = kb[col].dropna().unique()[:3]
    sample_str = str(list(sample))[:50]
    print(f"{col:<35} {missing:>10} {available_kb:>10}  {sample_str}")

Total common columns: 27

Attribute                           Missing D1   Avail KB  Sample values
------------------------------------------------------------------------------------------
brand                                       27       2084  ['Gigabyte', 'Western Digital', 'Corsair']
bus_type                                   184       1658  ['SATA III', 'PCIe 3.0 x4', 'USB 3.0']
chipset_name                               581        633  ['GeForce RTX 3080', 'GeForce GTX 1650', 'GeForce 
cluster_id                                   0       2200  [np.int64(1002037), np.int64(1004942), np.int64(10
color                                      692        245  ['Black', 'Silver', 'Cobalt Trim']
description                                  0       2200  ['Gigabyte NVIDIA GeForce RTX 3080 GAMING OC 10GB 
form_factor                                395       1110  ['3.5 Inch', 'M.2', '3.5-Inch']
height_mm                                  766        120  [np.float64(46.0), np.float64(10.5),

In [5]:
import pandas as pd

df1 = pd.read_json("normalized_products/dataset_1_normalized.json")
df2 = pd.read_json("normalized_products/dataset_2_normalized.json")
df3 = pd.read_json("normalized_products/dataset_3_normalized.json")
df4 = pd.read_json("normalized_products/dataset_4_normalized.json")
kb = pd.concat([df2, df3, df4], ignore_index=True)

def get_ground_truth(cluster_id, attribute, kb):
    matches = kb[kb["cluster_id"] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            return str(val).strip()
    return None

# Only check the interesting candidates
candidates = [
    "bus_type", "model_number", "model",
    "read_speed_mb_s", "write_speed_mb_s",
    "vram_gb", "storage_gb", "chipset_name",
    "memory_type", "form_factor", "interface_type",
    "storage_connection_type", "height_mm", "width_mm",
    "length_mm", "weight_g", "color"
]

print(f"{'Attribute':<30} {'Missing':>8} {'Recoverable':>12} {'Rate':>6}  Type")
print("-" * 70)
for attr in candidates:
    if attr not in df1.columns:
        continue
    missing_rows = df1[df1[attr].isna()]
    if len(missing_rows) == 0:
        print(f"{attr:<30} {'— no missing values —':>40}")
        continue
    recoverable = sum(
        1 for _, row in missing_rows.iterrows()
        if get_ground_truth(row["cluster_id"], attr, kb) is not None
    )
    rate = 100 * recoverable / len(missing_rows)
    dtype = "numeric" if df1[attr].dtype in ["float64", "int64"] else "text"
    print(f"{attr:<30} {len(missing_rows):>8} {recoverable:>12} {rate:>5.0f}%  {dtype}")

Attribute                       Missing  Recoverable   Rate  Type
----------------------------------------------------------------------
bus_type                            184          154    84%  text
model_number                        478          302    63%  text
model                                64           47    73%  text
read_speed_mb_s                     661          122    18%  numeric
write_speed_mb_s                    690           87    13%  numeric
vram_gb                             583            2     0%  numeric
storage_gb                          241            6     2%  numeric
chipset_name                        581            0     0%  text
memory_type                         602           43     7%  text
form_factor                         395           61    15%  text
interface_type                      376           64    17%  text
storage_connection_type             399           46    12%  text
height_mm                           766          101    13%

In [2]:
import pandas as pd

dfs = {
    1: pd.read_json("normalized_products/dataset_1_normalized.json"),
    2: pd.read_json("normalized_products/dataset_2_normalized.json"),
    3: pd.read_json("normalized_products/dataset_3_normalized.json"),
    4: pd.read_json("normalized_products/dataset_4_normalized.json"),
}

TARGET_ATTRIBUTES = [
    "bus_type", "model_number", "model",
    "read_speed_mb_s", "write_speed_mb_s",
    "vram_gb", "storage_gb", "chipset_name",
    "memory_type", "form_factor", "interface_type",
    "storage_connection_type", "height_mm", "width_mm",
    "length_mm", "weight_g", "color"
]

def get_ground_truth(cluster_id, attribute, kb):
    matches = kb[kb["cluster_id"] == cluster_id]
    for _, row in matches.iterrows():
        val = row.get(attribute)
        if pd.notna(val) and str(val).strip().lower() not in {"", "none", "nan"}:
            return str(val).strip()
    return None

for query_idx in [1, 2, 3, 4]:
    df_query = dfs[query_idx]
    kb = pd.concat([dfs[i] for i in [1,2,3,4] if i != query_idx], ignore_index=True)
    print(f"\n=== Query = Dataset {query_idx}, KB = others ===")
    print(f"{'Attribute':<25} {'Missing':>8} {'Recoverable':>12} {'Rate':>6}")
    print("-" * 55)
    for attr in TARGET_ATTRIBUTES:
        if attr not in df_query.columns:
            continue
        missing_rows = df_query[df_query[attr].isna()]
        if len(missing_rows) == 0:
            continue
        recoverable = sum(
            1 for _, row in missing_rows.iterrows()
            if get_ground_truth(row["cluster_id"], attr, kb) is not None
        )
        rate = 100 * recoverable / len(missing_rows)
        print(f"{attr:<25} {len(missing_rows):>8} {recoverable:>12} {rate:>5.0f}%")


=== Query = Dataset 1, KB = others ===
Attribute                  Missing  Recoverable   Rate
-------------------------------------------------------
bus_type                       184          154    84%
model_number                   478          302    63%
model                           64           47    73%
read_speed_mb_s                661          122    18%
write_speed_mb_s               690           87    13%
vram_gb                        583            2     0%
storage_gb                     241            6     2%
chipset_name                   581            0     0%
memory_type                    602           43     7%
form_factor                    395           61    15%
interface_type                 376           64    17%
storage_connection_type        399           46    12%
height_mm                      766          101    13%
width_mm                       767           84    11%
length_mm                      767           78    10%
weight_g                